# DocTamper 蒸馏评测（Colab）

使用 DTD reproduction 相同测试设置：`DocTamperV1-FCD` + `minq=75`。

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
%cd /content
if not os.path.exists('/content/DocTamper'):
    !git clone https://github.com/LeSiIence/DocTamper.git
%cd /content/DocTamper/models
!pip install -q lmdb albumentations segmentation_models_pytorch timm efficientnet_pytorch tqdm
!pip install -q opencv-python-headless Pillow google-cloud-storage
!git clone https://github.com/dwgoon/jpegio.git
%cd jpegio
!python setup.py install -q
%cd /content/DocTamper

In [ ]:
import os

# 把测试集放在 Drive 目录下，例如 /content/drive/MyDrive/TargetFolder/DocTamperV1-FCD
TARGET_FOLDER = '/content/drive/MyDrive/TargetFolder'
DATASET_NAME = 'DocTamperV1-FCD'
DATASET_SRC = f'{TARGET_FOLDER}/{DATASET_NAME}'
DATASET_DST = f'/content/DocTamper/{DATASET_NAME}'

if not os.path.exists(DATASET_DST):
    !cp -r "$DATASET_SRC" "$DATASET_DST"

assert os.path.exists(DATASET_DST), f'测试集不存在: {DATASET_DST}'
assert os.path.exists('/content/DocTamper/qt_table.pk'), '缺少 qt_table.pk'
assert os.path.exists('/content/DocTamper/pks'), '缺少 pks 目录'
os.makedirs('/content/DocTamper/pths', exist_ok=True)
print('测试集准备完成')

In [ ]:
import os
import re
from google.colab import auth
from google.cloud import storage

# 修改为你的 GCS 权重路径
DISTILL_CKPT_GS = 'gs://YOUR_BUCKET/YOUR_PATH/light_dtd_distill.pth'
LOCAL_CKPT = '/content/DocTamper/pths/light_dtd_distill_eval.pth'

auth.authenticate_user()
m = re.match(r'^gs://([^/]+)/(.+)$', DISTILL_CKPT_GS)
assert m is not None, f'无效的 GCS URI: {DISTILL_CKPT_GS}'
bucket_name, blob_name = m.group(1), m.group(2)
client = storage.Client()
bucket = client.bucket(bucket_name)
blob = bucket.blob(blob_name)
blob.download_to_filename(LOCAL_CKPT)
assert os.path.exists(LOCAL_CKPT)
print('权重下载完成:', LOCAL_CKPT)

In [ ]:
# 兼容补丁：某些版本 eval_light_dtd.py 缺少 import time
eval_file = '/content/DocTamper/models/eval_light_dtd.py'
with open(eval_file, 'r', encoding='utf-8') as f:
    txt = f.read()
if 'import time' not in txt:
    txt = 'import time\n' + txt
    with open(eval_file, 'w', encoding='utf-8') as f:
        f.write(txt)
print('eval_light_dtd.py ready')

%cd /content/DocTamper
!CUDA_VISIBLE_DEVICES=0 python -m models.eval_light_dtd --data_root /content/DocTamper/ --lmdb_name DocTamperV1-FCD --pth /content/DocTamper/pths/light_dtd_distill_eval.pth --minq 75 --batch_size 6 --num_workers 2